# CRAI base — statewide inputs (run rarely)

Builds the parts of the Cal-CRAI calculation that are **the same for every event and
location**, and caches them in `crai_cache/` for the event notebooks to read:

| Cached file | What it is |
|---|---|
| `adaptive_capacity.csv` | Built / governance / natural / society domain scores per tract |
| `climate_metrics_standardized.csv` | Every climate-domain metric, min-max standardized statewide |
| `ca_tracts.gpkg` | Statewide census tracts (EPSG:4326) |

`source` in **`configs/crai_base.yaml`** picks where these come from:

- **`local`** — the snapshot files in `crai_seed/` (seconds, no AWS). Same domain scores
  as the Sacramento 2020 post's CRAI layers.
- **`s3`** — a real refresh: download the Cal-CRAI metrics from
  `s3://ca-climate-index/3_fair_data/index_data/` and recompute every domain score.
  Needs AWS credentials; takes several minutes. Method copied from era-scratch
  `UPDATED_de_relevant_hazards_aci_layer.ipynb` (cells 4, 7, 9) and
  `UPDATED_DE_tool_hazard_layer.ipynb` (cells 3–5).

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import shutil
import sys
import tempfile

import geopandas as gpd
import pandas as pd
import yaml

# Shared code lives in helpers/ next to this notebook.
sys.path.insert(0, os.path.abspath("helpers"))
import crai_helpers as ch
from event_helpers import (DEFAULT_CRAI_CACHE, TRACTS_CACHE_FILE,
                           ADAPTIVE_CAPACITY_FILE, CLIMATE_METRICS_FILE, FRAMEWORK_DIR)

with open(os.path.join(FRAMEWORK_DIR, "configs", "crai_base.yaml")) as f:
    SETTINGS = yaml.safe_load(f)
SOURCE = SETTINGS["source"]
if SOURCE not in ("local", "s3"):
    raise ValueError(f"crai_base.yaml source must be 'local' or 's3', got {SOURCE!r}")


def framework_path(p):
    return p if os.path.isabs(p) or p.startswith("s3://") else os.path.join(FRAMEWORK_DIR, p)


CACHE_DIR = DEFAULT_CRAI_CACHE
os.makedirs(CACHE_DIR, exist_ok=True)
DOMAIN_COLS = ["governance_domain_score", "built_domain_score", "natural_domain_score", "society_domain_score"]
print(f"Building {CACHE_DIR} from source: {SOURCE}")

## 1. Statewide census tracts

Local mode reads `local.tracts`, falling back to the S3 shapefile if that file is missing
(it's git-ignored, so a fresh checkout won't have it).

In [ ]:
local_tracts = framework_path(SETTINGS["local"]["tracts"])
if SOURCE == "local" and os.path.exists(local_tracts):
    tracts = gpd.read_file(local_tracts)
else:
    if SOURCE == "local":
        print(f"{local_tracts} not found — reading tracts from S3 instead")
    tracts = gpd.read_file(SETTINGS["s3"]["tracts"])

tracts["GEOID"] = tracts["GEOID"].astype(str).str.zfill(11)
tracts = tracts[["GEOID", "geometry"]].to_crs("EPSG:4326")
tracts.to_file(os.path.join(CACHE_DIR, TRACTS_CACHE_FILE), driver="GPKG")
print(f"{len(tracts)} statewide tracts")

## 2a. Local: copy the snapshot domain scores and climate metrics

In [ ]:
if SOURCE == "local":
    adaptive_capacity = pd.read_csv(framework_path(SETTINGS["local"]["adaptive_capacity"]), dtype={"GEOID": str})
    climate_raw = pd.read_csv(framework_path(SETTINGS["local"]["climate_metrics"]), dtype={"GEOID": str})
    adaptive_capacity = adaptive_capacity[["GEOID"] + DOMAIN_COLS]
else:
    print("source is s3 — skipping")

## 2b. S3: download the Cal-CRAI metrics and recompute the domain scores

The CRAI classification helper *moves* files out of its input folder as it sorts them,
so this runs against a throwaway working copy of the download — the cached download
itself is never modified, and isn't re-downloaded unless `s3.redownload` is true.

In [ ]:
if SOURCE == "s3":
    s3 = SETTINGS["s3"]
    source_dir = os.path.join(CACHE_DIR, "source")
    meta_csv = framework_path(s3["metric_list"])
    # Written only after a full download, so an interrupted download is never
    # mistaken for a complete one.
    download_done = os.path.join(source_dir, ".download_complete")

    if s3.get("redownload") or not os.path.exists(download_done):
        if os.path.isdir(source_dir):
            shutil.rmtree(source_dir)
        ch.pull_csv_from_directory(s3["bucket"], s3["metrics_prefix"],
                                   output_folder=source_dir, search_zipped=False, print_name=False)
        open(download_done, "w").close()
    else:
        print(f"Reusing downloaded metrics in {source_dir}")

    work_dir = tempfile.mkdtemp(prefix="crai_base_")
    input_folder = os.path.join(work_dir, "input")
    shutil.copytree(source_dir, input_folder)

    domain_scores = {}
    for domain in ["built", "governance", "natural", "society"]:
        output_folder = os.path.join(work_dir, f"{domain}_folder")
        merged_output_file = os.path.join(work_dir, f"concatenate_{domain}_metrics.csv")

        metric_files, metric_dict = ch.create_metric_vulnerable_resilient_dict(
            domain, meta_csv, input_folder, output_folder
        )
        ch.process_domain_csv_files(domain, input_folder, output_folder, meta_csv, merged_output_file)

        df_to_run = pd.read_csv(merged_output_file)
        columns_to_process = [col for col in df_to_run.columns if col != "GEOID"]
        min_max_metrics = ch.min_max_standardize(df_to_run, columns_to_process)

        selected_columns = [col for col in min_max_metrics.columns if col == "GEOID" or "standardized" in col]
        standardized_df = min_max_metrics[selected_columns]

        # society-only override: energy burden metric isn't tagged vulnerable in the metric list
        if domain == "society":
            metric_dict["vulnerable"].append("energy_burden_ACS")

        vulnerable_columns_in_df = [
            col for col in standardized_df.columns if any(v in col for v in metric_dict["vulnerable"])
        ]
        adjusted_df = standardized_df.copy()
        adjusted_df.loc[:, vulnerable_columns_in_df] = 1 - adjusted_df.loc[:, vulnerable_columns_in_df]

        averaged_indicators = ch.compute_averaged_indicators(adjusted_df, metric_dict, print_summary=False)
        columns_to_sum = [col for col in averaged_indicators.columns if col != "GEOID"]
        summed_indicators = ch.compute_summed_indicators(averaged_indicators, columns_to_sum, domain + "_")
        columns_to_process = [col for col in summed_indicators.columns if col != "GEOID"]
        min_max_domain = ch.min_max_standardize(summed_indicators, columns_to_process)

        domain_scores[domain] = ch.calc_domain_score(min_max_domain, domain)
        print(f"{domain}: {len(domain_scores[domain])} tracts\n")

    adaptive_capacity = domain_scores["governance"]
    for domain in ["built", "natural", "society"]:
        adaptive_capacity = adaptive_capacity.merge(domain_scores[domain], on="GEOID", how="left")
    adaptive_capacity = adaptive_capacity[["GEOID"] + DOMAIN_COLS]

    climate_folder = os.path.join(work_dir, "climate_folder")
    merged_output_file = os.path.join(work_dir, "concatenate_climate_metrics.csv")
    ch.create_metric_vulnerable_resilient_dict("climate", meta_csv, input_folder, climate_folder)
    climate_raw = ch.process_domain_csv_files("climate", input_folder, climate_folder, meta_csv, merged_output_file)

    shutil.rmtree(work_dir)
else:
    print("source is local — skipping")

## 3. Write the cache

Climate metrics are min-max standardized statewide here in both modes. Each event
notebook picks its hazard's exposure/loss metrics from this table and adds its own
event metric.

In [ ]:
adaptive_capacity["GEOID"] = adaptive_capacity["GEOID"].astype(str).str.zfill(11)
adaptive_capacity.to_csv(os.path.join(CACHE_DIR, ADAPTIVE_CAPACITY_FILE), index=False)

climate = climate_raw.copy()
climate["GEOID"] = climate["GEOID"].astype(str).str.zfill(11)
climate = climate.sort_values(by="GEOID")
climate = ch.min_max_standardize(climate, [c for c in climate.columns if c != "GEOID"])
climate.to_csv(os.path.join(CACHE_DIR, CLIMATE_METRICS_FILE), index=False)

print(f"adaptive_capacity: {len(adaptive_capacity)} tracts, {adaptive_capacity[DOMAIN_COLS].isna().sum().sum()} missing scores")
print(f"climate metrics: {len(climate)} tracts")
print(sorted(c.replace("_min_max_standardized", "") for c in climate.columns if c.endswith("_min_max_standardized")))

In [ ]:
for f in sorted(os.listdir(CACHE_DIR)):
    print(f)